# Lab | Tools prompting — my solution

**Goal:** teach `phi3` (a small model with **no built-in tool calling**) to call tools anyway,
just by describing the tools in the prompt and asking for a JSON answer.

**My setup:** phi3 (~4B parameters, Microsoft) runs locally through **Ollama**, on the CPU only
(Dell Latitude, no GPU). Each model call takes roughly 20–30 seconds.
Ollama must be running in the background (`C:\Users\Windows\ollama\ollama.exe serve`).

## Step 0 — Quiet the deprecation warning

`langchain_community.llms.Ollama` still works, but LangChain prints two long warnings: the class is
*deprecated* (a newer one lives in the separate `langchain-ollama` package) and `langchain-community`
itself is being retired. The code still runs fine, so we hide those warnings to keep the outputs readable.

In [40]:
import warnings
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)
warnings.filterwarnings("ignore", message=".*langchain-community.*")

## Step 1 — Connect to phi3

`Ollama(model="phi3")` does not load anything yet — it just remembers *which* model to ask.
The first real call loads phi3 into memory (slow the first time), then answers.
This model is a plain **LLM**: you give it text, it gives back a string.

**Two limits (added after Step 6 got stuck):** phi3 doesn't know where its turn ends and sometimes keeps
inventing new conversation — on this CPU that ran for over 10 minutes per call. So we tell it:
- `stop=["System:", "Human:"]` — stop as soon as you start a new fake turn,
- `num_predict=200` — never write more than 200 tokens (a tool-call JSON needs ~40).

In [41]:
from langchain_community.llms import Ollama

# stop: cut the answer when phi3 starts inventing a new "System:"/"Human:" turn
# num_predict: never write more than 200 tokens (keeps each call short on this CPU)
model = Ollama(model="phi3", stop=["System:", "Human:"], num_predict=200)

print(model.invoke("Say hello in five words."))

Hello greetings


## Step 2 — Create the tools

A **tool** is just a normal Python function with the `@tool` decorator.
LangChain reads three things from it, and these are what the model will later see:
- the **name** (the function name),
- the **description** (the docstring),
- the **arguments** and their types (from the type hints).

In [42]:
from langchain_core.tools import tool


@tool
def multiply(x: float, y: float) -> float:
    """Multiply two numbers together."""
    return x * y


@tool
def add(x: int, y: int) -> int:
    "Add two numbers."
    return x + y


tools = [multiply, add]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
multiply
Multiply two numbers together.
{'x': {'title': 'X', 'type': 'number'}, 'y': {'title': 'Y', 'type': 'number'}}
--
add
Add two numbers.
{'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}


A tool can be run directly with `.invoke()` and a dictionary of arguments — no model involved yet.
This is exactly what our code will do later with the arguments the model picks.

In [43]:
multiply.invoke({"x": 4, "y": 5})

20.0

## Step 3 — Describe the tools to the model

phi3 cannot "see" Python functions. So we turn the tools into **plain text** with
`render_text_description` — one line per tool: name, arguments, description.

In [44]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

multiply(x: float, y: float) -> float - Multiply two numbers together.
add(x: int, y: int) -> int - Add two numbers.


Now the **system prompt**: it pastes that tool list in and tells the model exactly what to answer —
not the result, but *which tool to use and with which arguments*, as a JSON blob with `name` and `arguments`.

`ChatPromptTemplate` combines the system prompt with the user's question. `{input}` is a placeholder
that gets filled in each time we ask something.

In [45]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

### First try: what does phi3 answer?

`prompt | model` is a **chain**: the `|` pipes the filled-in prompt into the model.
We don't run any tool yet — we just look at the raw text phi3 sends back.

In [46]:
chain = prompt | model
message = chain.invoke({"input": "what's 3 plus 1132"})

# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

```json
{
  "name": "add",
  "arguments": {
    "x": 3,
    "y": 1132
  }
}
```





**What we saw:** phi3 picked the right tool (`add`) with the right arguments (`x=3`, `y=1132`),
wrapped in a ```json code block. But then it **kept talking**: it invented a new "System:" message
with made-up tools (`subtran`, `divide`) and a new question.

Why? `Ollama` here is a plain **LLM** (text completion), not a chat model. LangChain flattens the chat prompt
into one long text ("System: ... Human: ..."), and phi3 simply continues that text — it doesn't know where its turn ends.
The useful part is the first JSON block; the next step (the output parser) has to pull out just that.

## Step 4 — Add an output parser

phi3 answers with **text**. To run a tool we need a real Python **dictionary**:
`{"name": ..., "arguments": {...}}`.

`JsonOutputParser` does that conversion. It is forgiving: it looks for the first JSON block in the text
(also inside a \`\`\`json code fence), ignores what comes after it, and turns it into a dict.

We just add it as a third link to the chain: **prompt → model → parser**.

In [47]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()
chain.invoke({"input": "what's thirteen times 4"})

{'name': 'multiply', 'arguments': {'x': 13, 'y': 4}}

**What we saw:** the output is now a Python dict, e.g. `{'name': 'multiply', 'arguments': {'x': 13, 'y': 4}}`.

phi3 is not consistent in *how* it writes the JSON. In three test runs I got three different shapes:
1. JSON in a \`\`\`json block, followed by an invented conversation (like in Step 3),
2. a sentence first ("Here is the requested tool usage…") and then the JSON block,
3. just the bare JSON.

The parser handled all three and returned the same dict. 🎉
We have now taught the model how to **request** a tool — next we write the code that actually **runs** it.

## Step 5 — Actually run the tool 🏃

So far the model only **asks** for a tool. Now we write the code that **runs** it.

`invoke_tool` takes the dict from the parser and does three simple things:
1. builds a lookup table `{"multiply": multiply, "add": add}` from our `tools` list,
2. picks the tool whose name the model asked for,
3. calls it with the model's arguments (`.invoke(arguments)`).

`ToolCallRequest` is only a **type hint**: it documents that the input must have `name` and `arguments` keys.
`config` is optional LangChain plumbing (callbacks, metadata) that we just pass along.

In [48]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

### Test it 🧪 (no model involved)

We hand-write the dict the model *would* produce. This runs instantly, because phi3 is not asked anything.

In [49]:
invoke_tool({"name": "multiply", "arguments": {"x": 3, "y": 5}})

15.0

**What we saw:** `15.0` — the function found `multiply` by its name and ran it with `x=3, y=5`.
(It is `15.0` and not `15` because `multiply` is typed with `float`.)

## Step 6 — Put it all together

Now the full chain, four links:

**prompt** (question + tool list) → **phi3** (writes JSON text) → **JsonOutputParser** (text → dict) → **invoke_tool** (runs the tool)

You ask in plain English and get the computed number back. Note the question writes "thirteen" as a **word** —
phi3 has to turn it into `13` itself.

In [50]:
chain = prompt | model | JsonOutputParser() | invoke_tool
chain.invoke({"input": "what's thirteen times 4.14137281"})

53.83784653

## Step 7 — Return the tool inputs too

The answer alone (`53.84…`) doesn't show *how* we got it. It's useful to also see which tool the model picked
and with which arguments — e.g. to check its work.

`RunnablePassthrough.assign(output=invoke_tool)` keeps the parser's dict (`name`, `arguments`) and **adds** one more key,
`output`, holding the tool's result.

In [51]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "what's thirteen times 4.14137281"})

{'name': 'multiply',
 'arguments': {'x': 13, 'y': 4.14137281},
 'output': 53.83784653}

**What we saw:** in my 3 test runs phi3 always returned
`{'name': 'multiply', 'arguments': {'x': 13, 'y': 4.14137281}, 'output': 53.83784653}` —
the right tool, "thirteen" correctly turned into `13`, and the exact result (13 × 4.14137281 = 53.83784653).

That's the end of the worked demo. Next comes **"Your turn"**: 3 tools of our own.

<hr>

# 🚀 Your turn — 3 new tools

We replace `add` and `multiply` with three tools that are easy to check by hand:

| Tool | Argument | What it does |
|---|---|---|
| `celsius_to_fahrenheit` | `celsius: float` | converts a temperature |
| `count_words` | `text: str` | counts the words in a text |
| `reverse_string` | `text: str` | writes a text backwards |

Two of them take the same kind of input (a `text`), so the model has to read the **descriptions** to choose.

## 1 — Design the 3 tools

Same recipe as before: a normal function, `@tool` on top, a clear docstring (that's what the model reads), and type hints.

In [52]:
@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from degrees Celsius to degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


@tool
def count_words(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())


@tool
def reverse_string(text: str) -> str:
    """Reverse a piece of text so it reads backwards, letter by letter."""
    return text[::-1]


# Quick check without the model
print(celsius_to_fahrenheit.invoke({"celsius": 100}))
print(count_words.invoke({"text": "one two three"}))
print(reverse_string.invoke({"text": "abc"}))

212.0
3
cba


## 2 — Update the tools list and check what the model will see

`invoke_tool` from Step 5 looks tools up in the global `tools` list, so replacing the list is enough — no change to `invoke_tool` needed.

💡 Tip from the lab: print `rendered_tools` to double-check the descriptions.

In [53]:
tools = [celsius_to_fahrenheit, count_words, reverse_string]

rendered_tools = render_text_description(tools)
print(rendered_tools)

celsius_to_fahrenheit(celsius: float) -> float - Convert a temperature from degrees Celsius to degrees Fahrenheit.
count_words(text: str) -> int - Count how many words are in a piece of text.
reverse_string(text: str) -> str - Reverse a piece of text so it reads backwards, letter by letter.


## 3 — Regenerate the prompt

The system prompt is an **f-string**: `{rendered_tools}` was filled in at the moment we created it.
So the old prompt still lists `multiply` and `add` — we must build it again.

In [54]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

## 4 — Rebuild the chain

Same chain as Step 7 (it shows the tool, the arguments **and** the result), just built from the new prompt.

We also add a tiny helper `ask()` that prints the result — or the error, instead of crashing the notebook.
That way one bad answer from phi3 doesn't stop the other questions.

In [55]:
chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)


def ask(question):
    print("Q:", question)
    try:
        print("->", chain.invoke({"input": question}))
    except Exception as e:
        print("-> ERROR:", type(e).__name__, e)
    print()

## 5 — Put it to the test

Two questions per tool — one plain, one a bit trickier. Each question takes ~15–35 seconds.

In [56]:
ask("How warm is 25 degrees Celsius in Fahrenheit?")
ask("It's minus 10 degrees outside. What is that in Fahrenheit?")

Q: How warm is 25 degrees Celsius in Fahrenheit?
-> {'name': 'celsius_to_fahrenheit', 'arguments': {'celsius': 25}, 'output': 77.0}

Q: It's minus 10 degrees outside. What is that in Fahrenheit?
-> {'name': 'celsius_to_fahrenheit', 'arguments': {'celsius': -10}, 'output': 14.0}



In [57]:
ask("How many words are in 'the quick brown fox jumps over the lazy dog'?")
ask("Count the words in this sentence: I am learning tool calling with a small local model")

Q: How many words are in 'the quick brown fox jumps over the lazy dog'?
-> {'name': 'count_words', 'arguments': {'text': 'the quick brown fox jumps over the lazy dog'}, 'output': 9}

Q: Count the words in this sentence: I am learning tool calling with a small local model
-> {'name': 'count_words', 'arguments': {'text': 'I am learning tool calling with a small local model'}, 'output': 10}



In [58]:
ask("Write 'Ironhack' backwards")
ask("Reverse the text: hello world")

Q: Write 'Ironhack' backwards
-> {'name': 'reverse_string', 'arguments': {'text': 'Ironhack'}, 'output': 'kcahnorI'}

Q: Reverse the text: hello world
-> {'name': 'reverse_string', 'arguments': {'text': 'hello world'}, 'output': 'dlrow olleh'}



**Expected answers:** 25 °C = 77 °F, −10 °C = 14 °F · 9 words, 10 words · `kcahnorI`, `dlrow olleh`

## 6 — Reflect

*(My test run, before writing this cell. phi3 is random, so your run may differ — note what **you** see.)*

**What went right (5 of 6):** phi3 always chose the right tool, even between the two `text` tools.
It turned "minus 10 degrees" into `-10`, and it copied the sentences into `text` exactly — so the word counts (9, 10)
and the reversed strings were correct.

**What went wrong (1 of 6):** for *"How warm is 25 degrees Celsius in Fahrenheit?"* phi3 picked the right tool but
**misspelled its name**: `celsiius_to_fahrenheit` (double *i*). Our lookup `tool_name_to_tool[name]` is an exact
dictionary match, so it failed with `KeyError: 'celsiius_to_fahrenheit'`.

**Why:** phi3 is a small model that writes the name **token by token** — it doesn't copy it, it re-generates it.
A long name made of several pieces (`celsius` + `_to_` + `fahrenheit`) gives more chances to slip.
And phi3 was never fine-tuned for tool calling, so nothing forces it to use only the exact names we listed.
It samples with some randomness, which is why the same kind of question worked the second time.

**How we could fix it:**
- shorter, simpler tool names (e.g. `c_to_f`),
- `temperature=0` in `Ollama(...)` so the answers are less random,
- a few-shot example in the system prompt showing the exact JSON (lab bonus idea),
- error handling in `invoke_tool`: if the name doesn't exist, send the error back to the model and ask it to try again
  (lab bonus idea) — or pick the closest existing name.